In [1]:
# --- SETUP: paths + robust pairing + I/O helpers (run once) ---
from pathlib import Path
import re, os, json, math
import numpy as np
import nibabel as nib
from scipy.ndimage import gaussian_filter, zoom, rotate, shift, binary_closing, generate_binary_structure, label

# SOURCE hires test set (single-folder mode with images + masks)
SRC_DIR   = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
# Where to place all degraded datasets
OUT_ROOT  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# --------- pairing (mirrors your training loader’s spirit) ----------
def is_mask_name(name: str) -> bool:
    n = name.lower()
    return ("mask" in n) or ("lesion" in n)

def strip_ext(name: str) -> str:
    return name[:-7] if name.endswith(".nii.gz") else os.path.splitext(name)[0]

def norm_key(name: str) -> str:
    # remove known suffixes to match img<->mask
    stem = strip_ext(name)
    # drop common endings
    for sfx in ["_T1w_MNI_norm","_T1w_MNI","_T1w_brain","_T1w","_T1","_image","_img","_img_prepped"]:
        if stem.endswith(sfx): stem = stem[: -len(sfx)]
    for sfx in ["_lesion_mask_MNI_clean","_lesion_mask_MNI","_lesion_mask","_desc-lesion_mask","_mask","_mask_prepped"]:
        if stem.endswith(sfx): stem = stem[: -len(sfx)]
    return stem.rstrip("_")

# build maps
imgs, msks = {}, {}
for p in sorted(SRC_DIR.glob("*.nii.gz")):
    (msks if is_mask_name(p.name) else imgs)[norm_key(p.name)] = p

keys = sorted(set(imgs) & set(msks))
pairs = [(imgs[k], msks[k]) for k in keys]
print(f"Found pairs: {len(pairs)}  (imgs={len(imgs)}, msks={len(msks)})")

# --------- helpers ----------
def load_nii(p: Path) -> nib.Nifti1Image:
    return nib.load(str(p))

def data_f32(img: nib.Nifti1Image) -> np.ndarray:
    return np.asarray(img.get_fdata(dtype=np.float32), dtype=np.float32)

def save_like(ref_img: nib.Nifti1Image, array: np.ndarray, out_path: Path, dtype=None):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = (array.astype(dtype) if dtype is not None else array.astype(np.float32))
    nii = nib.Nifti1Image(arr, ref_img.affine, ref_img.header.copy())
    nib.save(nii, str(out_path))

def save_and_update_spacing(ref_img: nib.Nifti1Image, array: np.ndarray, out_path: Path, new_spacing_xyz):
    """Use ref affine but update pixdim to reflect a new voxel size (mm)."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    hdr = ref_img.header.copy()
    hdr["pixdim"][1:4] = np.array(new_spacing_xyz, dtype=np.float32)
    nii = nib.Nifti1Image(array.astype(np.float32), ref_img.affine, hdr)
    nib.save(nii, str(out_path))

def resample_factor(arr, factors, order):
    """Resample by 1/factors using scipy.zoom (anti-alias BEFORE calling)."""
    # zoom expects output/input; if factor=2 (coarsen), zoom=1/2
    zf = tuple(1.0/f for f in factors)
    return zoom(arr, zf, order=order, prefilter=True)

def ensure_uint8_mask(b):
    return (b > 0).astype(np.uint8)

def percentile_scale(vol, pmin=1, pmax=99):
    lo, hi = np.percentile(vol, [pmin, pmax])
    hi = max(hi, lo + 1e-6)
    x = np.clip((vol - lo) / (hi - lo), 0, 1)
    return x


Found pairs: 138  (imgs=138, msks=138)


# 1) Crude downsampling (no anti-alias) — “worst-case decimation”

Why / real-world: Stress-tests robustness to botched resampling pipelines that skip anti-aliasing, producing jaggies and aliasing. Rare, but if your preproc ever fails, this is what it looks like.

What the code does

Picks a stride in each axis (e.g., factors=(2,2,5)).

Subsamples the 3D array by taking every factor[k]-th voxel along each axis:
xr = x[::fx, ::fy, ::fz]
No smoothing/anti-aliasing beforehand.

Updates pixdim (voxel spacing) by multiplying the original spacing by the same factors so geometry stays honest.

Mask is downsampled with the exact same stride and saved as uint8 (nearest-like behavior due to slicing).

What this mimics

A worst-case resampling pipeline where images were naively decimated without low-pass filtering.

You’ll see jagged edges (“stair-step”), aliasing, and partial volume artifacts amplified.

Why it’s useful

It stress-tests robustness to sloppy preprocessing you might encounter in the wild (legacy scripts or mangled exports).

If your model holds up here, it’ll be fine on any sane downsampling.

Caveats

This is harsher than what clinics do; vendors apply proper reconstruction filtering.

In [2]:
# --- 1) CRUDE DOWNSAMPLING (no anti-alias), e.g., 2x in-plane, 5x through-plane ---
OUT_DIR = OUT_ROOT / "test_hires_crude_2x2x5x"
factors = (2, 2, 5)  # (X,Y,Z) coarsening factors

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img); y = data_f32(msk)

    # crude decimation: just take every nth voxel
    xr = x[::factors[0], ::factors[1], ::factors[2]]
    yr = y[::factors[0], ::factors[1], ::factors[2]]  # masks follow grid

    # update voxel spacing (pixdim) by factors
    pix = img.header.get_zooms()[:3]
    new_pix = (pix[0]*factors[0], pix[1]*factors[1], pix[2]*factors[2])

    base = img_p.stem.replace("_T1w","")
    save_and_update_spacing(img, xr, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz", new_pix)
    save_and_update_spacing(msk, ensure_uint8_mask(yr), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", new_pix)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_crude_2x2x5x


# 2) Thick slices (Z-only downsample with anti-alias)

Why / real-world: Very common: 1×1×5 mm or similar to shorten scans. Causes strong partial-volume in Z.

What the code does

Applies a Gaussian blur along Z only: gaussian_filter(x, sigma=(0,0,sigma_z)).

Then decimates in Z by an integer factor (e.g., 5): xr = xb[:, :, ::5].

Updates only the Z voxel spacing: new_dz = old_dz * 5.

Mask is decimated slice-wise using nearest (simple stride). No smoothing is applied to masks to keep labels crisp.

What this mimics

Common protocol trade-off: keep in-plane high (e.g., ~1×1 mm) but use thick slices (e.g., 5 mm) to shorten scan time.

Real images have through-plane blur and partial volume: structures smaller than the slice thickness smear into neighbors.

Why it’s useful

Many routine T1/T2/FLAIR stacks are anisotropic. Models trained on 1 mm isotropic can struggle here; this tests that gap.

Caveats

True slice thickness combines slice profile + gaps; here we simulate the net blur/gross thickness (a good approximation).

In [3]:
# --- 2) THICK SLICES: blur in Z then 5x decimate (keep XY) ---
OUT_DIR = OUT_ROOT / "test_hires_thickslice_1x1x5mm"
factor_z = 5
sigma_z_vox = 2.0  # pre-blur along Z only

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img); y = data_f32(msk)

    xb = gaussian_filter(x, sigma=(0,0,sigma_z_vox))
    xr = xb[:, :, ::factor_z]
    yr = y[:, :, ::factor_z]  # nearest in Z by simple slice step

    pix = img.header.get_zooms()[:3]
    new_pix = (pix[0], pix[1], pix[2]*factor_z)

    base = img_p.stem.replace("_T1w","")
    save_and_update_spacing(img, xr, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz", new_pix)
    save_and_update_spacing(msk, ensure_uint8_mask(yr), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", new_pix)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_thickslice_1x1x5mm


# 3) In-plane coarsening (XY downsample; Z intact)

Why / real-world: Protocols that keep thin slices but coarsen in-plane to reduce time. Tests sensitivity to small in-plane structures.

What the code does

Applies a Gaussian blur in X/Y: gaussian_filter(x, sigma=(sigma_xy, sigma_xy, 0)).

Decimates in X/Y by an integer factor (e.g., 2): xr = xb[::2, ::2, :].

Updates only X/Y spacing; Z spacing unchanged.

Mask is downsampled with nearest in X/Y by the same stride.

What this mimics

Protocols that keep thin slices but reduce the in-plane matrix (e.g., 256→128) to cut scan time or extend coverage.

Small cortical or juxtacortical lesions get “blockier” and less distinct in-plane.

Why it’s useful

Tests sensitivity to in-plane resolution loss separately from slice thickness effects.

Caveats

Real recon often uses vendor-specific filters; our Gaussian + decimate is a principled, reproducible stand-in.

In [4]:
# --- 3) IN-PLANE COARSENING: 2x in X/Y with anti-alias; Z intact ---
OUT_DIR = OUT_ROOT / "test_hires_inplane_2x2x1mm"
factor_xy = 2
sigma_xy = 1.0

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img); y = data_f32(msk)

    xb = gaussian_filter(x, sigma=(sigma_xy, sigma_xy, 0))
    xr = xb[::factor_xy, ::factor_xy, :]  # decimate XY
    # masks nearest-neighbor in XY
    yr = y[::factor_xy, ::factor_xy, :]

    pix = img.header.get_zooms()[:3]
    new_pix = (pix[0]*factor_xy, pix[1]*factor_xy, pix[2])

    base = img_p.stem.replace("_T1w","")
    save_and_update_spacing(img, xr, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz", new_pix)
    save_and_update_spacing(msk, ensure_uint8_mask(yr), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", new_pix)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_inplane_2x2x1mm


# 4) Reduced SNR (Rician noise)

Why / real-world: Fewer averages / higher acceleration → noisier magnitude images.

What the code does

Estimates a robust “signal” level (e.g., 90th percentile).

Chooses noise standard deviation sigma = signal / TARGET_SNR.

Adds two independent zero-mean Gaussian noise fields (real and imaginary), then converts to magnitude:
xn = sqrt((x + n_real)^2 + n_imag^2).

Geometry unchanged; masks are copied as-is.

What this mimics

In magnitude MR, noise is Rician (not Gaussian). Lower SNR arises with fewer averages, higher acceleration, smaller voxels, or lower field strength.

SNR roughly scales with voxel volume and sqrt(NEX) (number of excitations).

Why it’s useful

Many clinical scans (especially fast ones) are noisy. This checks whether your model’s normalization and learned priors carry you through.

Caveats

True coil/multiple-receive-channel effects alter noise distribution spatially; this uses a homogeneous approximation.

In [5]:
# --- 4) RICIAN NOISE: choose σ to target SNR≈15 (or change TARGET_SNR) ---
OUT_DIR = OUT_ROOT / "test_hires_rician_snr15"
TARGET_SNR = 15.0  # ~typical clinic range 10–20

rng = np.random.default_rng(123)
for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img)

    # robust "signal" estimate: 90th percentile in-brain-ish
    sig = np.percentile(x, 90)
    sigma = max(sig / TARGET_SNR, 1e-6)

    nr = rng.normal(0, sigma, size=x.shape).astype(np.float32)
    ni = rng.normal(0, sigma, size=x.shape).astype(np.float32)
    xn = np.sqrt((x + nr)**2 + ni**2).astype(np.float32)

    base = img_p.stem.replace("_T1w","")
    save_like(img, xn, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz", dtype=np.float32)
    # geometry unchanged → copy mask bytes
    save_like(msk, data_f32(msk), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_rician_snr15


# 5) Slice-wise motion (rigid jitter)

Why / real-world: Patient motion during 2D acquisitions → slice-to-slice misalignment/blur.

What the code does

For each slice, applies a small random rotation (±a few degrees) and pixel shift (±a few px):
rotate(..., order=1) for image, order=0 for mask; then shift(...) similarly.

This produces slice-to-slice misalignments and slight blurring/ghosting from interpolation.

Voxel spacing unchanged; just geometry perturbations.

What this mimics

2D multi-slice acquisitions where the patient moves between slice excitations → slice stack doesn’t line up perfectly.

Very common in restless patients, pediatrics, or longer scans.

Why it’s useful

Motion is one of the biggest real-world degraders. Even tiny rotations/shift destroy fine boundaries and create zebra-like slice seams.

Caveats

Real motion can be continuous and within-TR; this is a discrete per-slice model (captures the dominant visual effect).

In [6]:
# --- 5) SLICE-WISE MOTION JITTER: small per-slice rotations/shifts ---
OUT_DIR = OUT_ROOT / "test_hires_motion_slicejitter"
rng = np.random.default_rng(7)
deg_range = 2.0   # ± degrees
px_range  = 2.0   # ± pixels

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img); y = data_f32(msk)
    H, W, Z = x.shape
    xm = np.empty_like(x); ym = np.empty_like(y)

    for k in range(Z):
        ang = rng.uniform(-deg_range, deg_range)
        dx  = rng.uniform(-px_range,  px_range)
        dy  = rng.uniform(-px_range,  px_range)
        # rotate then shift (order 1 for image, 0 for mask)
        sl  = rotate(x[:,:,k], angle=ang, reshape=False, order=1, mode="nearest")
        sl  = shift(sl, shift=(dy, dx), order=1, mode="nearest")
        xm[:,:,k] = sl
        slm = rotate(y[:,:,k], angle=ang, reshape=False, order=0, mode="nearest")
        slm = shift(slm, shift=(dy, dx), order=0, mode="nearest")
        ym[:,:,k] = slm

    base = img_p.stem.replace("_T1w","")
    save_like(img, xm, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    save_like(msk, ensure_uint8_mask(ym), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_motion_slicejitter


# 6) k-space undersampling aliasing (zero-filled recon)

Why / real-world: Aggressive parallel imaging / partial-Fourier → residual alias/ghost when reconstruction is imperfect.

What the code does

For each slice (2D FFT), it builds a 1D sampling mask along phase-encode (Y) with:

a fully sampled DC/center band (low frequencies),

and randomly selected outer lines to hit a target acceleration (e.g., 2×).

Sets unacquired lines to zero (no fancy reconstruction), then inverse FFT back.

Geometry unchanged; masks are copied.

What this mimics

Parallel imaging / compressed sensing when acceleration is aggressive or reconstruction fails → residual aliasing/ghosts.

The center band models vendor practice of keeping low-freq fully sampled to preserve contrast.

Why it’s useful

Tests robustness to coherent artifacts (ghosting/alias) that look nothing like Gaussian noise but happen in fast protocols.

Caveats

Real recon uses sensitivity maps and iterative solvers; this is the pessimistic zero-filled limit (harder than reality).

In [7]:
# --- 6) K-SPACE UNDER-SAMPLING: 2x along phase-encode (Y), VD with center fully sampled ---
OUT_DIR = OUT_ROOT / "test_hires_kspace_alias_x2"
rng = np.random.default_rng(42)
accel = 2.0
center_frac = 0.12  # fully-sampled DC region

def undersample_mask(ny, accel, center_frac=0.12):
    m = np.zeros(ny, bool)
    c = int(ny*center_frac/2)
    mid = ny//2
    m[mid-c:mid+c+1] = True
    # pick the rest uniformly at rate 1/accel to reach target
    want = int(round(ny/accel)) - m.sum()
    idxs = [i for i in range(ny) if not m[i]]
    rng.shuffle(idxs)
    m[idxs[:max(want,0)]] = True
    return m

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img)
    H, W, Z = x.shape

    # FFT per slice in XY
    xm = np.empty_like(x)
    for k in range(Z):
        sl = x[:,:,k]
        ksl = np.fft.fftshift(np.fft.fft2(sl))
        mask = undersample_mask(W, accel, center_frac)
        ksl[:, ~mask] = 0
        slr = np.real(np.fft.ifft2(np.fft.ifftshift(ksl)))
        xm[:,:,k] = slr.astype(np.float32)

    base = img_p.stem.replace("_T1w","")
    save_like(img, xm, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    # geometry unchanged → copy mask
    save_like(msk, data_f32(msk), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_kspace_alias_x2


# 7) Gibbs ringing (k-space truncation)

Why / real-world: Limited high-frequency sampling yields ringing near edges (very common on sharp T1).

What the code does

For each slice, FFT to k-space, keep only the central radius (e.g., 75% of k-space energy) and zero the rest.

Inverse FFT back → sharp transitions overshoot/undershoot (ringing near edges).

Geometry unchanged; masks are copied.

What this mimics

Limited high-frequency sampling or apodization issues → Gibbs ringing (very visible on T1 edges).

Also a stand-in for too-aggressive smoothing at recon time.

Why it’s useful

Ringing creates false positives along tissue boundaries and confuses edge-based features learned by CNNs.

Caveats

Real systems use windowing; we emulate a clean radial crop to make the effect consistent and tunable

In [8]:
# --- 7) GIBBS RINGING: crop high-frequency k-space radius per slice ---
OUT_DIR = OUT_ROOT / "test_hires_gibbs"
keep_radius = 0.75  # keep 75% of k-space radius

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img)
    H, W, Z = x.shape
    xm = np.empty_like(x)

    cy, cx = H//2, W//2
    R2 = (min(H, W) * keep_radius / 2.0)**2

    for k in range(Z):
        sl = x[:,:,k]
        ksl = np.fft.fftshift(np.fft.fft2(sl))
        yy, xx = np.ogrid[:H, :W]
        mask = (yy-cy)**2 + (xx-cx)**2 <= R2
        ksl[~mask] = 0
        xm[:,:,k] = np.real(np.fft.ifft2(np.fft.ifftshift(ksl))).astype(np.float32)

    base = img_p.stem.replace("_T1w","")
    save_like(img, xm, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    save_like(msk, data_f32(msk), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_gibbs


# 8) Bias-field inhomogeneity (smooth multiplicative field)

Why / real-world: Receive/B1 inhomogeneity → slow intensity roll-off; can fool intensity normalization.

What the code does

Generates a smooth 3D low-frequency field f(x,y,z) (Gaussian-filtered noise), scaled to be around 1±amp (e.g., ±30%).

Multiplies the image: xb = x * f.

Geometry unchanged; masks are copied.

What this mimics

Receive coil and B1 inhomogeneity → slow intensity roll-off across the head. Stronger at 3T+, surface coils, or large FOVs.

Can break naive intensity normalization and thresholding.

Why it’s useful

Checks whether your preprocessing (e.g., z-score/percentile norm) and the model are tolerant to broad intensity drifts.

Caveats

True bias fields can correlate with anatomy and coil layout; we simulate a generic smooth pattern.

In [9]:
# --- 8) BIAS-FIELD: multiply by smooth low-frequency 3D field ---
OUT_DIR = OUT_ROOT / "test_hires_biasfield"
rng = np.random.default_rng(99)
amp = 0.3  # ±30%
sigma = (40, 40, 20)  # smoothness (vox)

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img)
    # build smooth field ~1±amp
    f = gaussian_filter(rng.normal(0,1,size=x.shape).astype(np.float32), sigma=sigma)
    f = (f - f.min()) / (f.max() - f.min() + 1e-6)
    f = 1.0 + amp*(2*f - 1.0)
    xb = (x * f).astype(np.float32)

    base = img_p.stem.replace("_T1w","")
    save_like(img, xb, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    save_like(msk, data_f32(msk), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_biasfield


# 9) Bit-depth / quantization (8-bit)

Why / real-world: Legacy exports or lossy preprocessing → banding and reduced dynamic range.

What the code does

Scales intensities robustly into [0,1] using image percentiles (e.g., p1–p99).

Quantizes to 256 levels (uint8) and writes that dtype into the NIfTI header.

Geometry unchanged; masks are copied.

What this mimics

Legacy PACS exports, lossy conversion, or “normalized” research datasets that were clamped/quantized.

Banding and loss of subtle gray/white contrast.

Why it’s useful

Tests whether your model truly needs subtle intensity resolution, or if it’s robust to coarse discretization.

Caveats

Real pipelines may apply window/level before quantization; we use percentile scaling to remain dataset-agnostic.

In [ ]:
# --- 9) QUANTIZATION to 8-bit using robust scaling (p1–p99) ---
OUT_DIR = OUT_ROOT / "test_hires_quant8"

# Fix saving for quantization (or any write-out) — robust .nii.gz handling
from pathlib import Path
import numpy as np
import nibabel as nib

# --- paths ---
HIRES_DIR = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
OUT_DIR   = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_quant8")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def niigz_stem(p: Path) -> str:
    """Return filename without the .nii.gz (or last) extension."""
    name = p.name
    if name.endswith(".nii.gz"):
        return name[:-7]
    return p.stem  # fallback

def save_like(ref_path: Path, array: np.ndarray, out_path: Path, dtype=None):
    ref = nib.load(str(ref_path))
    data = np.ascontiguousarray(array.astype(dtype if dtype is not None else np.float32))
    hdr = ref.header.copy()
    nib.save(nib.Nifti1Image(data, ref.affine, hdr), str(out_path))

def robust_uint8_quant(x, p_lo=1, p_hi=99):
    x = np.asarray(x, np.float32)
    lo, hi = np.percentile(x, [p_lo, p_hi])
    if hi <= lo:  # degenerate fallback
        lo, hi = x.min(), x.max() if x.max() > x.min() else (0.0, 1.0)
    y = np.clip((x - lo) / (hi - lo), 0, 1)
    return (y * 255.0 + 0.5).astype(np.uint8)

# find pairs in single-folder layout
imgs = sorted(HIRES_DIR.glob("*_T1w_MNI_norm.nii.gz"))
masks = {niigz_stem(p).replace("_lesion_mask_MNI_clean",""): p
         for p in HIRES_DIR.glob("*_lesion_mask_MNI_clean.nii.gz")}

pairs = []
for img in imgs:
    stem = niigz_stem(img).replace("_T1w_MNI_norm", "")
    msk = masks.get(stem)
    if msk is not None:
        pairs.append((img, msk))

print(f"Found pairs: {len(pairs)}")

for img, msk in pairs:
    # load, quantize to uint8
    ni = nib.load(str(img)); x = ni.get_fdata(dtype=np.float32)
    x8 = robust_uint8_quant(x, p_lo=1, p_hi=99)

    # build sane basenames
    base = niigz_stem(img).replace("_T1w_MNI_norm", "")  # e.g., 'sub-XXX_ses-YYY'
    img_out = OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    msk_out = OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    # write image as uint8 (preserve geometry)
    hdr_img = ni.header.copy(); hdr_img.set_data_dtype(np.uint8)
    nib.save(nib.Nifti1Image(x8, ni.affine, hdr_img), str(img_out))

    # write mask (copy as-is; ensure uint8)
    nm = nib.load(str(msk)); y = nm.get_fdata(dtype=np.float32)
    save_like(msk, (y > 0).astype(np.uint8), msk_out, dtype=np.uint8)

print("Done:", OUT_DIR)


FileNotFoundError: [Errno 2] No such file or directory: '/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_quant8/sub-M2029_ses-180_MNI_norm.nii_T1w_MNI_norm.nii.gz'

# 10) Slice gaps (simulate interleaved acquisition)

Why / real-world: Older 2D protocols with gaps reduce cross-slice context without changing voxel size metadata.

What the code does

Every Nth slice (e.g., 1 in 5), it replaces that slice with a neighbor (keeps array size the same).

This introduces repeated slices and missing anatomical information, mimicking a “gap” without changing pixdim.

Mask slices are duplicated at the same indices.

What this mimics

Older or fast 2D protocols with slice gaps (e.g., 4 mm slice with 1 mm gap). You don’t see anatomy for the skipped positions.

Downstream resampling to isotropic can smear/duplicate planes.

Why it’s useful

A nasty corner case for 3D models: they expect consistent context along Z; gaps violate that assumption.

Caveats

True gaps change physical spacing; here we simulate the visual effect while keeping array dimensions constant for easy downstream processing.

In [ ]:
# --- 10) SLICE GAPS: remove every Nth slice and duplicate neighbor to keep size ---
OUT_DIR = OUT_ROOT / "test_hires_slicegap_20pct"
gap_every = 5  # drop 1 in 5 (~20%)

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img); y = data_f32(msk)
    H, W, Z = x.shape
    xm = x.copy(); ym = y.copy()

    for k in range(gap_every-1, Z, gap_every):
        # replace slice k with neighbor (simulate a visual gap)
        src = max(0, k-1)
        xm[:,:,k] = xm[:,:,src]
        ym[:,:,k] = ym[:,:,src]

    base = img_p.stem.replace("_T1w","")
    save_like(img, xm, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    save_like(msk, ensure_uint8_mask(ym), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)
